# Cyberbullying Detection using BERTweet
This notebook trains a BERTweet model to classify tweets into 6 categories:
- not_cyberbullying
- gender
- religion
- other_cyberbullying
- age
- ethnicity

## Cell 1: Install and Import Libraries

In [ ]:
!pip install transformers torch emoji pandas scikit-learn seaborn matplotlib -q

import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import emoji
import json
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from collections import defaultdict

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 8.2 MB/s eta 0:00:00
Running on: cpu


## Cell 2: Load Dataset

In [ ]:
from google.colab import files
import io

print("Upload cyberbullying_tweets.csv:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))
print(f"✅ Loaded {len(df)} rows")
print(df.head())

## Cell 3: Preprocess Text
- Converts emojis to text (e.g. 😊 becomes 'smiling face')
- Replaces URLs with HTTPURL token
- Replaces @mentions with @USER token
- These normalizations match how BERTweet was originally trained

In [ ]:
USE_EMOJIS = True

def process_text(text):
    text = str(text)
    # Convert emojis to text e.g. 😊 becomes 'smiling face'
    if USE_EMOJIS:
        text = emoji.demojize(text, delimiters=(" ", " "))
        text = text.replace("_", " ").replace(":", "")
    else:
        text = emoji.replace_emoji(text, replace='')
    # Normalize URLs and mentions for BERTweet
    text = re.sub(r"http\S+|www\S+|https\S+", 'HTTPURL', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '@USER', text)
    return re.sub(r'\s+', ' ', text).strip()

df['content'] = df['tweet_text'].apply(process_text)
print("✅ Text preprocessed")
print(df['content'].head())

## Cell 4: Create Labels and Split Data
- Maps text labels to numbers (e.g. 'religion' -> 0, 'gender' -> 1)
- Splits data into 90% train, 5% validation, 5% test

In [ ]:
# Map text labels to numbers
class_names = df['cyberbullying_type'].unique()
label_map = {name: i for i, name in enumerate(class_names)}
df['sentiment'] = df['cyberbullying_type'].map(label_map)

print("Classes:", class_names)
print("Label map:", label_map)
print("\nClass distribution:")
print(df['cyberbullying_type'].value_counts())

# Split into train (90%), val (5%), test (5%)
RANDOM_SEED = 42
df_train, df_test = train_test_split(df, test_size=0.1, random_state=RANDOM_SEED)
df_val, df_test = train_test_split(df_test, test_size=0.5, random_state=RANDOM_SEED)

print(f"\nTrain: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

## Cell 5: Load BERTweet Tokenizer
BERTweet is a BERT model pre-trained on 850M English tweets

In [ ]:
PRE_TRAINED_MODEL_NAME = 'vinai/bertweet-base'
tokenizer = AutoTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME, use_fast=False)
print("✅ BERTweet tokenizer loaded!")

## Cell 6: Dataset and DataLoader
- SentimentDataset converts raw text into tokenized tensors for the model
- DataLoader batches the data for efficient training

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, reviews, targets, tokenizer, max_len):
        self.reviews = reviews
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, item):
        review = str(self.reviews[item])
        target = self.targets[item]
        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'review_text': review,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(target, dtype=torch.long)
        }

BATCH_SIZE = 8
MAX_LEN = 128

train_data_loader = DataLoader(
    SentimentDataset(df_train['content'].to_numpy(), df_train['sentiment'].to_numpy(), tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, num_workers=0
)
val_data_loader = DataLoader(
    SentimentDataset(df_val['content'].to_numpy(), df_val['sentiment'].to_numpy(), tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, num_workers=0
)
test_data_loader = DataLoader(
    SentimentDataset(df_test['content'].to_numpy(), df_test['sentiment'].to_numpy(), tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, num_workers=0
)
print("✅ DataLoaders ready!")

## Cell 7: Model Architecture
- Uses BERTweet as the base model
- Adds a Dropout layer to prevent overfitting
- Adds a linear layer to output 6 class scores
- Softmax converts scores to probabilities

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, n_classes):
        super(SentimentClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(PRE_TRAINED_MODEL_NAME)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, input_ids, attention_mask):
        _, pooled_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False
        )
        output = self.drop(pooled_output)
        output = self.out(output)
        return self.softmax(output)

model = SentimentClassifier(len(class_names))
model = model.to(device)
print("✅ Model ready!")

## Cell 8: Training Setup
- AdamW optimizer with learning rate 2e-5 (standard for fine-tuning BERT)
- Linear scheduler gradually reduces learning rate during training
- CrossEntropyLoss is used for multi-class classification

In [ ]:
EPOCHS = 4
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_data_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
loss_fn = nn.CrossEntropyLoss().to(device)
print("✅ Training setup ready!")

## Cell 9: Train and Evaluate Functions

In [ ]:
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model.train()
    losses = []
    correct_predictions = 0
    for d in data_loader:
        input_ids = d['input_ids'].to(device)
        attention_mask = d['attention_mask'].to(device)
        targets = d['targets'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, targets)
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    return correct_predictions.double() / n_examples, np.mean(losses)

def eval_model(model, data_loader, loss_fn, device, n_examples):
    model.eval()
    losses = []
    correct_predictions = 0
    with torch.no_grad():
        for d in data_loader:
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            targets = d['targets'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, targets)
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
    return correct_predictions.double() / n_examples, np.mean(losses)

print("✅ Functions ready!")

## Cell 10: Training Loop
- Trains for 4 epochs
- Saves the best model based on validation accuracy

In [ ]:
history = defaultdict(list)
best_accuracy = 0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)
    train_acc, train_loss = train_epoch(model, train_data_loader, loss_fn, optimizer, device, scheduler, len(df_train))
    print(f'Train loss {train_loss:.4f} accuracy {train_acc:.4f}')
    val_acc, val_loss = eval_model(model, val_data_loader, loss_fn, device, len(df_val))
    print(f'Val   loss {val_loss:.4f} accuracy {val_acc:.4f}')
    print()
    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)
    if val_acc > best_accuracy:
        torch.save(model.state_dict(), 'best_model_state.bin')
        best_accuracy = val_acc

# Save class names for later use
with open('class_names.json', 'w') as f:
    json.dump(list(class_names), f)

print(f"✅ Training done! Best accuracy: {best_accuracy:.4f}")
print("✅ Model and class names saved!")

## Cell 11: Plot Training History

In [ ]:
# Convert tensors to float for plotting
train_acc = [x.item() if hasattr(x, 'item') else x for x in history['train_acc']]
val_acc = [x.item() if hasattr(x, 'item') else x for x in history['val_acc']]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## Cell 12: Confusion Matrix
Shows how well the model classifies each category

In [ ]:
def get_predictions(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            targets = d['targets'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)

preds, labels = get_predictions(model, test_data_loader)

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=class_names))

## Cell 13: Classify New Sentences
Test the model on your own sentences

In [ ]:
def classify_text(text):
    text = process_text(text)
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    with torch.no_grad():
        output = model(input_ids=input_ids, attention_mask=attention_mask)
    _, prediction = torch.max(output, dim=1)
    label = class_names[prediction.item()]
    confidence = output[0][prediction.item()].item()
    return label, confidence

# Test sentences
test_sentences = [
    "You are so stupid because of your religion 🕌",
    "Have a great day everyone! 😊🎉",
    "People of that race are all the same 🤮",
    "Happy birthday! Hope you have a wonderful day 🎂🎈",
    "You are too old to be doing this 👴",
    "Go back to your country 🖕",
    "Love and respect for all ❤️🌍",
    "Your gender makes you worthless 🤡",
]

print("=" * 60)
for sentence in test_sentences:
    label, confidence = classify_text(sentence)
    print(f"Text:       {sentence}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2%}")
    print("-" * 60)